# 02 - Scheduled Briefings: promotion that curates, remembers where it came
from, and runs without you

Notebook 01 fixed how the harness learns *procedures*. This notebook fixes the
other promotion path - scratch notes graduating into long-term memory - which the
review summarised as *"promotion without curation, growth without forgetting"*:

| Review gap (Path 1) | This notebook |
|---|---|
| Everything graduates within the hour | **Opt-in** `/inbox` + an LLM **triage gate** (keep / discard / hold) |
| 200-word chunks re-extracted by the LLM | **Whole-note promotion**, verbatim (`extract_memories=False`) |
| No provenance, nothing supersedes | A **queryable sidecar** (source path, revision, supersession links) |
| Consumer never actually scheduled | In-DB **DBMS_SCHEDULER** producer + drainable consumer + **queue-depth health check** |
| (bonus) persistence-of-one-fact only | **Continuity of work**: a new session resumes the briefing *plan*, not just facts |

The payoff: a **morning asset-risk briefing** the database assembles on schedule -
the LLM narrates the numbers; it never invents them.

## 0 · Setup

Connect, verify notebook 01's artifacts, check privileges, stand up the embedder.

In [ ]:
import array
import json
import uuid

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model, litellm_spec
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import promote

settings = load_settings()
conn = get_connection(settings)
require(conn, "01_self_improving_copilot")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

In [ ]:
# The scheduled pipeline needs two privileges most ADB users lack by default.
with conn.cursor() as cur:
    cur.execute("SELECT privilege FROM user_sys_privs")
    _privs = {r[0] for r in cur.fetchall()}
_missing = {"CREATE PROCEDURE", "CREATE JOB"} - _privs
check(not _missing,
      f"scheduler privileges present (run as ADMIN if missing: "
      + "; ".join(f'GRANT {p} TO {settings.db_user}' for p in sorted(_missing)) + ")"
      if _missing else "scheduler privileges present (CREATE PROCEDURE, CREATE JOB)")

In [ ]:
import asyncio
import numpy as np
from langchain_oracledb.embeddings.oracleai import OracleEmbeddings
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name):
        self.provider = OracleEmbeddings(
            conn=conn, params={"provider": "database", "model": model_name}
        )

    def embed(self, texts, *, is_query=False):
        return np.asarray(self.provider.embed_documents(list(texts)), dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


embedder = OracleONNXEmbedder(conn, settings.embed_model)
check(embedder.embed(["hello"]).shape == (1, 384), "in-database embedder ready")

## 1 · The Agent Memory SDK - configured for *whole-note* promotion

`extract_memories=False` is a deliberate review fix, not a shortcut: the original
pipeline chunked files into 200-word windows and had the LLM *re-interpret* each one
at ingest - confident misreadings of mid-sentence fragments became durable facts.
Here a curated note is stored **verbatim** as one memory; the triage gate (next
section) is where judgement happens, *before* anything is durable.

In [ ]:
from oracleagentmemory.core import OracleAgentMemory, SchemaPolicy
from oracleagentmemory.core.llms.llm import Llm as SdkLlm

memory = OracleAgentMemory(
    connection=conn,
    embedder=embedder,
    llm=SdkLlm(**litellm_spec(settings)),
    extract_memories=False,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="CITY_",
)

# Round-trip smoke test: store, find, delete.
_mid = memory.add_memory("smoke test memory - safe to delete",
                         memory_type="fact", user_id="_smoke", agent_id="CITY")
_hits = memory.search("smoke test", user_id="_smoke", agent_id="CITY", max_results=1)
check(len(_hits) == 1 and _hits[0].id == _mid, "SDK add_memory/search round-trip")
memory.delete_memory(_mid)
ok("OracleAgentMemory ready (CITY_ tables, whole-note mode)")

## 2 · The scratchpad: cheap to write, safe to be wrong in

A plain table posing as a tiny filesystem. Inspectors jot notes all day; only
`/inbox/<asset>/...` is a promotion *candidate area*. Working files - drafts,
abandoned queries, tool dumps - live outside it and are **never** promoted, which
restores the scratchpad's contract the review showed the old pipeline destroyed.

In [ ]:
def _table_exists(name):
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = :t", t=name)
        return cur.fetchone()[0] > 0


if not _table_exists("HARNESS_SCRATCH"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_SCRATCH (
            path       VARCHAR2(512) PRIMARY KEY,
            content    CLOB NOT NULL,
            revision   VARCHAR2(64) NOT NULL,
            status     VARCHAR2(1) DEFAULT 'N'
                       CHECK (status IN ('N','S','P','D','H')),
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
    conn.commit()


def write_note(path, content):
    rev = promote.note_revision(content)
    with conn.cursor() as cur:
        cur.execute("DELETE FROM HARNESS_SCRATCH WHERE path = :p", p=path)
        cur.execute("""INSERT INTO HARNESS_SCRATCH (path, content, revision, status)
                       VALUES (:p, :c, :r, 'N')""", p=path, c=content, r=rev)
    conn.commit()
    return rev


def read_note(path):
    with conn.cursor() as cur:
        cur.execute("SELECT content, revision, status FROM HARNESS_SCRATCH WHERE path = :p",
                    p=path)
        row = cur.fetchone()
    if row is None:
        return None
    content = row[0].read() if hasattr(row[0], "read") else row[0]
    return {"path": path, "content": content, "revision": row[1], "status": row[2]}


def list_notes(prefix="/"):
    with conn.cursor() as cur:
        cur.execute("""SELECT path, status FROM HARNESS_SCRATCH
                       WHERE path LIKE :p ORDER BY path""", p=prefix + "%")
        return cur.fetchall()


def set_note_status(path, status):
    with conn.cursor() as cur:
        cur.execute("UPDATE HARNESS_SCRATCH SET status = :s WHERE path = :p",
                    s=status, p=path)
    conn.commit()

ok("scratch store ready")

In [ ]:
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
_asset_names = sorted({x["asset_name"] for x in _logs})
ASSET_A = "Harbor Bridge" if "Harbor Bridge" in _asset_names else _asset_names[0]

# Seed the inbox with real field narratives (non-routine first - richer signal),
# and the review's own cautionary trio OUTSIDE the inbox.
_candidates = [l for l in _logs if l["asset_name"] == ASSET_A and l["severity"] != "routine"]
_candidates += [l for l in _logs if l["asset_name"] == ASSET_A and l["severity"] == "routine"]
INBOX_SEED = []
for i, log in enumerate(_candidates[:4]):
    path = f"/inbox/{ASSET_A}/{i}.md"
    INBOX_SEED.append((path, log["narrative"], log["days_ago"]))
    write_note(path, log["narrative"])

SUPERSEDE_DEFECT_PATH = f"/inbox/{ASSET_A}/defect-bearing.md"
SUPERSEDE_REPAIR_PATH = f"/inbox/{ASSET_A}/repair-bearing.md"
write_note(SUPERSEDE_DEFECT_PATH,
           f"Bearing plates at pier 2 of {ASSET_A} show active corrosion with "
           "measurable section loss. Load rating review recommended; monitor monthly.")

write_note("/work/draft_notes.md",
           "TODO: check if the load rating includes the retrofit?? probably not")
write_note("/work/attempt1_wrong.sql",
           "SELECT asset, COUNT(*) FROM findings GROUP BY severity  -- wrong, abandoned")
write_note("/tool_out/similar_dump.json", json.dumps({"rows": list(range(50))}))

for p, s in list_notes("/"):
    print(f"{s}  {p}")
ok(f"{len(INBOX_SEED) + 1} inbox candidates, 3 working files (outside the inbox)")

## 3 · The triage gate: judgement *before* durability

Two filters, in order:

1. **Opt-in area** - anything outside `/inbox/` is discarded without spending a
   model call (the review's `/work/attempt1_wrong.sql` can never leak into memory)
2. **LLM triage** - inbox notes are classified `keep` (durable, factual, worth
   retrieval attention), `discard` (ephemeral, speculative, duplicative), or
   `hold` (incomplete - stays in scratch, re-triaged next cycle)

In [ ]:
from pydantic import BaseModel
from typing import Literal


class Triage(BaseModel):
    decision: Literal["keep", "discard", "hold"]
    reason: str


_triage_model = CHAT.with_structured_output(Triage)

TRIAGE_PROMPT = (
    "You are the curation gate for a city-infrastructure agent's long-term memory.\n"
    "Everything stored durably competes for retrieval attention forever, so be picky.\n"
    "keep    = a completed observation or action worth recalling months from now\n"
    "          (equipment facts, completed work, hazards, measured conditions).\n"
    "discard = ephemeral, speculative, draft-like, or content-free.\n"
    "hold    = potentially valuable but incomplete or unresolved (open questions,\n"
    "          pending measurements).\n\n"
    "Note path: {path}\nNote content:\n{content}"
)


def triage_note(path, content):
    # ✏️ TODO(1): the opt-in gate comes FIRST and never costs a model call.
    #             If promote.triage_allowed(path) is False, return
    #             Triage(decision='discard', reason='outside the /inbox opt-in area').
    #   ... your code here ...
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    return _triage_model.invoke(TRIAGE_PROMPT.format(path=path, content=content), config=cfg)


for p, _s in list_notes("/"):
    note = read_note(p)
    t = triage_note(p, note["content"])
    print(f"{t.decision:8s} {p}\n         {t.reason[:100]}")
ok("triage gate live")

## 4 · Promotion: whole notes, with a paper trail

A kept note is stored **verbatim** as one memory. Its provenance - source path,
content revision, observation age - lands in a **sidecar table** you can query with
SQL, not a blob you can't. And before storing, a supersession check asks: does this
note *replace* something we already believe? (A repair note supersedes the defect it
fixed - the review's 'archaeology site' problem, closed.)

In [ ]:
if not _table_exists("HARNESS_MEMORY_META"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE HARNESS_MEMORY_META (
            memory_id     VARCHAR2(128) PRIMARY KEY,
            asset_id      VARCHAR2(128) NOT NULL,
            source_path   VARCHAR2(512) NOT NULL,
            revision      VARCHAR2(64) NOT NULL,
            days_ago      NUMBER DEFAULT 0,
            superseded_by VARCHAR2(128),
            promoted_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
    conn.commit()


def asset_from_path(path):
    parts = path.split("/")
    return parts[2] if len(parts) > 3 and parts[1] == "inbox" else None


class Supersedes(BaseModel):
    supersedes: bool
    reason: str


_supersede_model = CHAT.with_structured_output(Supersedes)


def _check_supersession(content, candidate):
    """candidate: (memory_id, existing_content). True if the new note makes the
    old one obsolete (repair completes a defect, a measurement replaces an estimate)."""
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    q = ("Existing memory:\n{old}\n\nNew note:\n{new}\n\n"
         "Does the new note make the existing memory OBSOLETE (completed repair,\n"
         "corrected fact, replaced measurement)? Additional detail alone does not\n"
         "supersede.").format(old=candidate[1], new=content)
    return _supersede_model.invoke(q, config=cfg)


def promote_note(path):
    note = read_note(path)
    t = triage_note(path, note["content"])
    if t.decision == "discard":
        set_note_status(path, "D")
        return None
    if t.decision == "hold":
        set_note_status(path, "H")
        return None
    asset_id = asset_from_path(path)
    # ✏️ TODO(2): supersession check - search this asset's existing memories
    #             (memory.search(note content, user_id=asset_id, agent_id='CITY',
    #             max_results=3)), skip any already superseded (sidecar lookup),
    #             and collect ids the LLM says are made obsolete.
    #   ... your code here ...
    # ✏️ TODO(3): whole-note promotion - store the note VERBATIM as one memory
    #             (memory.add_memory, memory_type='fact', user_id=asset_id,
    #             agent_id='CITY'), record provenance in the sidecar, link
    #             superseded ids, and mark the scratch note 'P'.
    #   ... your code here ...
    return mid

ok("promotion pipeline defined (triage -> supersession -> whole-note store)")